In [1]:
import logging
from exp.run import ExperimentRun, SummarySectionName
from exp.config import TransformerExperiments, CNNExperiments
logging.basicConfig(level=logging.ERROR)

In [2]:
config = TransformerExperiments()
config.repeats = 5
config.gpu_id = 1

# config = CNNExperiments()
exp = ExperimentRun(config=config)

In [3]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch built with CUDA version: {torch.version.cuda}")
print(f"CUDA version: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")

PyTorch version: 2.6.0+cu124
PyTorch built with CUDA version: 12.4
CUDA version: True
CUDA device count: 2


In [4]:
models = [
    "EleutherAI/gpt-neo-125M",
    "facebook/opt-125m",
    "facebook/opt-350m",
    "cerebras/Cerebras-GPT-111M",
    "microsoft/deberta-base",
    "T5-small",
    "t5-base",
    "distilbert/distilgpt2",
    "openai-community/gpt2",
]
model = models[1]
batch = 10
optimizer = "AdamW"
gpu_id = 1
task_id = None
in_docker = True

In [5]:
for model in models:
    for batch in range(10, 15, 5):
        for i in range(config.repeats):
            exp.add_task(
                model_name=model,
                batch_size=batch,
                optimizer=optimizer,
                gpu_id=gpu_id,
                task_id=task_id,
            )
# for batch in range(5, 20, 5):
#     for i in range(1):
#         exp.add_task(
#             model_name=model,
#             batch_size=batch,
#             optimizer=optimizer,
#             gpu_id=gpu_id,
#             task_id=task_id,
#         )
# exp.add_task(
#     model_name=model,
#     batch_size=batch,
#     optimizer=optimizer,
#     gpu_id=gpu_id,
#     task_id=task_id,
# )



## Measure Ground Truth and Estimated Memory for Each job

In [ ]:
exp.run_group_truth(in_docker=in_docker)

100%|██████████| 135/135 [00:01<00:00, 69.57it/s]


=============== Start massively run for GPU train ======================


  0%|          | 0/135 [00:00<?, ?it/s]

## Estimate Max GPU Memory by DNNmem

In [7]:
exp.run_estimation(
    estimators=[
        SummarySectionName.DNNmem,
        SummarySectionName.LLmem,
        SummarySectionName.schedtune
    ],
    in_docker=in_docker
)

================== Create docker containers ==================


100%|██████████| 9/9 [00:18<00:00,  2.05s/it]


================== Execute docker containers ==================
=============== Start massively run for GPU train ======================


100%|██████████| 9/9 [01:36<00:00, 10.69s/it]

================== Statistics ==================
Run(success/total): 27/27


In [24]:
exp.verify_llmem_result()

100%|██████████| 9/9 [01:02<00:00,  6.91s/it]


## Estimate Max GPU Memory by SchedTune

In [5]:
exp.statistics()
results = exp.to_evaluation_result()

100%|██████████| 9/9 [00:00<00:00, 11893.11it/s]


=============== Statistics for Transformer-Exp ==================
train: 9/9
config: 9/9
groundtruth: 9/9
solution: 9/9
schedtune: 8/9
DNNmem: 8/9
LLmem: 9/9


100%|██████████| 9/9 [00:00<00:00, 3679.93it/s]
